In [1]:
%pip install numpy gymnasium torch



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Define the PFSP environment
import numpy as np

# Dummy problem: sequence of job IDs; return "makespan" (lower is better)
def calculate_makespan(sequence):
    # Simulate cost: sum of job ID differences
    return sum(abs(sequence[i] - sequence[i+1]) for i in range(len(sequence)-1))


In [ ]:
# Define the Genetic Algorithm
def initialize_population(size, num_jobs):
    return [np.random.permutation(num_jobs).tolist() for _ in range(size)]

def crossover(parent1, parent2):
    cut = np.random.randint(1, len(parent1)-1)
    child = parent1[:cut] + [j for j in parent2 if j not in parent1[:cut]]
    return child

def mutate(sequence, mutation_rate):
    if np.random.rand() < mutation_rate:
        i, j = np.random.randint(0, len(sequence), size=2)
        sequence[i], sequence[j] = sequence[j], sequence[i]
    return sequence


In [ ]:
## Define the RL agent
import torch
import torch.nn as nn
import torch.optim as optim

class QNetwork(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, output_size)
        )

    def forward(self, x):
        return self.net(x)

class RLAgent:
    def __init__(self, state_dim, action_dim, lr=0.001, gamma=0.9):
        self.q_net = QNetwork(state_dim, action_dim)
        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.gamma = gamma
        self.actions = [(0.1, 'roulette'), (0.3, 'roulette'), (0.1, 'tournament'), (0.3, 'tournament')]

    def get_state(self, fitness_scores):
        return torch.tensor([
            np.mean(fitness_scores),
            np.std(fitness_scores),
            np.min(fitness_scores)
        ], dtype=torch.float32)

    def choose_action(self, state, epsilon=0.1):
        if np.random.rand() < epsilon:
            return np.random.randint(len(self.actions))
        with torch.no_grad():
            q_values = self.q_net(state)
        return torch.argmax(q_values).item()

    def update(self, state, action, reward, next_state):
        q_vals = self.q_net(state)
        with torch.no_grad():
            next_q_vals = self.q_net(next_state)
        target = reward + self.gamma * torch.max(next_q_vals)
        loss = (q_vals[action] - target) ** 2
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()


In [6]:
num_jobs = 10
pop_size = 20
generations = 50

agent = RLAgent(state_dim=3, action_dim=4)
population = initialize_population(pop_size, num_jobs)

for gen in range(generations):
    fitness = [calculate_makespan(ind) for ind in population]
    state = agent.get_state(fitness)
    action_idx = agent.choose_action(state)
    mutation_rate, selection_type = agent.actions[action_idx]

    # Selection
    if selection_type == 'roulette':
        probs = 1 / (np.array(fitness) + 1e-6)
        probs /= probs.sum()
        parents = [population[np.random.choice(len(population), p=probs)] for _ in range(pop_size)]
    elif selection_type == 'tournament':
        parents = []
        for _ in range(pop_size):
            group = np.random.choice(population, size= 3)
            best = min(group, key=calculate_makespan)
            parents.append(best)

    # Crossover + Mutation
    offspring = []
    for i in range(0, len(parents), 2):
        child = crossover(parents[i], parents[i+1])
        child = mutate(child, mutation_rate)
        offspring.append(child)

    # Evaluate new population
    new_fitness = [calculate_makespan(ind) for ind in offspring]
    reward = np.mean(fitness) - np.mean(new_fitness)  # reward = fitness gain
    next_state = agent.get_state(new_fitness)

    # RL update
    agent.update(state, action_idx, reward, next_state)

    population = offspring
    print(f"Gen {gen+1} | Avg Makespan: {np.mean(new_fitness):.2f} | Action: {agent.actions[action_idx]}")


Gen 1 | Avg Makespan: 30.70 | Action: (0.3, 'roulette')
Gen 2 | Avg Makespan: 28.20 | Action: (0.3, 'roulette')
Gen 3 | Avg Makespan: 27.00 | Action: (0.3, 'roulette')
Gen 4 | Avg Makespan: 26.80 | Action: (0.3, 'roulette')
Gen 5 | Avg Makespan: 27.50 | Action: (0.3, 'roulette')
Gen 6 | Avg Makespan: 30.90 | Action: (0.3, 'roulette')
Gen 7 | Avg Makespan: 29.60 | Action: (0.3, 'roulette')
Gen 8 | Avg Makespan: 30.30 | Action: (0.3, 'roulette')
Gen 9 | Avg Makespan: 24.50 | Action: (0.3, 'roulette')
Gen 10 | Avg Makespan: 24.50 | Action: (0.3, 'roulette')
Gen 11 | Avg Makespan: 28.30 | Action: (0.3, 'roulette')
Gen 12 | Avg Makespan: 28.70 | Action: (0.3, 'roulette')
Gen 13 | Avg Makespan: 26.90 | Action: (0.3, 'roulette')
Gen 14 | Avg Makespan: 25.80 | Action: (0.3, 'roulette')
Gen 15 | Avg Makespan: 27.40 | Action: (0.3, 'roulette')
Gen 16 | Avg Makespan: 25.50 | Action: (0.1, 'roulette')
Gen 17 | Avg Makespan: 24.40 | Action: (0.3, 'roulette')
Gen 18 | Avg Makespan: 24.50 | Action: (